## DATA CAPTURE FROM CAMERA

• CREATE CAMERA CLASS.   
• INITIALISE SETUP.    
• CREATE CAPTURE FUNCTION.    
• SAVE VIDEO CAPTURED.   

Import the motors module for robot movement 

In [3]:
from robot_utils import (
    crop_frame,
    detect_yellow,
    fit_line,
    get_line_direction,
    get_steering_magnitude,
    get_rope_side
)

In [ ]:
import ipywidgets.widgets as widgets
import motors

robot = motors.MotorsYukon(mecanum=False)
print("Robot is ready:)")

## CAMERA CLASS 

initialise camera    

Core functions:  

• capture frames: get the .avi and .bin files this way      
• start thread       
• stop thread    


In [ ]:
import traitlets
import cv2
import numpy as np
import pyzed.sl as sl
import math
import numpy as np
import sys
import math
import threading
from traitlets.config.configurable import SingletonConfigurable
import time

import ipywidgets.widgets as widgets
from IPython.display import display

#create two widgets for the displaying of the image
display_color = widgets.Image(format='jpeg', width='45%') #determine the width of the color image
display_depth = widgets.Image(format='jpeg', width='45%')  #determine the width of the depth image
layout=widgets.Layout(width='100%')

sidebyside = widgets.HBox([display_color, display_depth],layout=layout) #horizontal display


# display the widget
display(sidebyside) 

timestamp = time.strftime('%Y%m%d_%H%M%S')
# Define a Camera class that inherits from SingletonConfigurable
class Camera(SingletonConfigurable):
    color_value = traitlets.Any() # monitor the color_value variable
    def __init__(self):
        super(Camera, self).__init__()

        self.zed = sl.Camera()
        # Create a InitParameters object and set configuration parameters
        init_params = sl.InitParameters()
        init_params.camera_resolution = sl.RESOLUTION.VGA #VGA(672*376), HD720(1280*720), HD1080 (1920*1080) or ...
        init_params.depth_mode = sl.DEPTH_MODE.NONE  # depth not needed

        # Open the camera
        status = self.zed.open(init_params)
        if status != sl.ERROR_CODE.SUCCESS: #Ensure the camera has opened succesfully
            print("Camera Open : "+repr(status)+". Exit program.")
            self.zed.close()
            exit(1)

         # Create and set RuntimeParameters after opening the camera
        self.runtime = sl.RuntimeParameters()

        #flag to control the thread
        self.thread_runnning_flag = False

        # Get the height and width
        camera_info = self.zed.get_camera_information()
        self.width = camera_info.camera_configuration.resolution.width
        self.height = camera_info.camera_configuration.resolution.height
        self.image = sl.Mat(self.width,self.height,sl.MAT_TYPE.U8_C4, sl.MEM.CPU)

        #setup output file
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        self.color_writer = cv2.VideoWriter(f'color_video{timestamp}.avi', fourcc, 30, (672, 376))

    def _capture_frames(self): #For data capturing only

        while(self.thread_runnning_flag==True): #continue until the thread_runnning_flag is set to be False
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                
                # Retrieve Left image
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)

                self.color_value = self.image.get_data()
                self.color_value = cv2.cvtColor(self.color_value, cv2.COLOR_BGRA2BGR)

                 # Save to file
                try:
                    self.color_writer.write(self.color_value)
                except:
                    print("Error writing file")
                    
    def start(self): #start the data capture thread
        if self.thread_runnning_flag == False: #only process if no thread is running yet
            self.thread_runnning_flag=True #flag to control the operation of the _capture_frames function
            self.thread = threading.Thread(target=self._capture_frames) #link thread with the function
            self.thread.start() #start the thread

    def stop(self): #stop the data capture thread
        if self.thread_runnning_flag == True:
            self.color_writer.release()
            self.thread_runnning_flag = False #exit the while loop in the _capture_frames
            self.thread.join() #wait the exiting of the thread       

def bgr8_to_jpeg(value):#convert numpy array to jpeg coded data for displaying 
    return bytes(cv2.imencode('.jpg',value)[1])
    
#create a camera object
camera = Camera()
camera.start() # start capturing the data

#Convert a NumPy array to JPEG-encoded data for display
def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg',value)[1])


In [ ]:
# Link camera feed to display widget
camera.observe(lambda change: update_display(change['new']), names=['color_value'])

def update_display(frame):
    if frame is not None:
        display_color.value = bgr8_to_jpeg(frame)

In [ ]:
from collections import deque 
import cv2

def run_line_only(robot,
                  base_speed           = 0.30,
                  spin_speed           = 0.10,
                  micro_threshold      = 0.10,
                  micro_frames         = 2,
                  max_spin_frames      = 35,
                  align_threshold      = 0.20,
                  consecutive_required = 9):

    score_history         = deque(maxlen=micro_frames)
    consecutive_agreement = 0
    align_count           = 0
    in_corner             = False
    corner_frames         = 0
    last_turn             = None
    normal_frames         = 0
    last_rope_side        = None

    print("Starting — line fit only")

    try:
        while True:
            t_start = time.time()

            frame = camera.color_value
            if frame is None:
                continue
            if frame.shape[2] == 4:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGRA2BGR)

            # --- Geometry only ---
            #crops frame
            cropped   = crop_frame(frame)
            # detects yellow
            mask      = detect_yellow(cropped)
            #fits line
            line      = fit_line(mask)
            #determines direction and magnitude, and rope side with most pixels
            magnitude = get_steering_magnitude(line, cropped.shape)
            direction, score, _ = get_line_direction(line, mask, cropped.shape)
            score_history.append(magnitude)
            rope_side = get_rope_side(mask)

            #last recorded rope side for fallback
            if rope_side is not None:
                last_rope_side = rope_side

            fallback = rope_side or last_rope_side or last_turn or 'right'

            # --- Consecutive agreement — line direction + rope_side ---
            if rope_side is not None and \
               direction in ('left', 'right') and \
               rope_side == direction:
                consecutive_agreement += 1
                last_turn = direction
            else:
                consecutive_agreement = 0

            # --- Normal frame counter ---
            if not in_corner:
                normal_frames += 1
            else:
                normal_frames = 0

            # --- Corner trigger ---
            corner_trigger = (
                consecutive_agreement >= consecutive_required and
                not in_corner
            )

            # ============================================================
            # STATE
            # ============================================================
            final = 'straight'
            #initiate corner state if not already in one and trigger conditions met
            if not in_corner and corner_trigger:
                in_corner             = True
                corner_frames         = 0
                align_count           = 0
                consecutive_agreement = 0
                print(f"CORNER → {last_turn}")

            if in_corner:
                corner_frames += 1
                final = last_turn

                #waits for good alignment for a few frames before exiting corner state
                if abs(magnitude) < align_threshold:
                    align_count += 1
                    if align_count >= 3:
                        in_corner     = False
                        align_count   = 0
                        normal_frames = 0
                        last_turn     = None
                        print("EXIT corner")
                else:
                    align_count = 0

                #fallback if spins for too long without good alignment
                if corner_frames > max_spin_frames:
                    in_corner     = False
                    align_count   = 0
                    normal_frames = 0
                    print(f"SPIN TIMEOUT | fallback:{fallback}")

                # Motor command
                if last_turn == 'left':
                    robot.spinLeft(speed=spin_speed)
                elif last_turn == 'right':
                    robot.spinRight(speed=spin_speed)
                else:
                    robot.stop()

            else:
                #straight section behavuour
                #micro-correction to align robot to left or right
                recent = list(score_history)
                if len(recent) == micro_frames and \
                   all(s < -micro_threshold for s in recent):
                    robot.left(speed=base_speed * 0.8)
                elif len(recent) == micro_frames and \
                     all(s > micro_threshold for s in recent):
                    robot.right(speed=base_speed * 0.8)
                else:
                    robot.forward(speed=base_speed)

            print(f"LINE_ONLY | {'CORNER' if in_corner else 'NORMAL'} | "
                  f"dir:{direction} | mag:{magnitude:+.3f} | "
                  f"rope:{rope_side} | agree:{consecutive_agreement} | "
                  f"last_turn:{last_turn}")

            elapsed = time.time() - t_start
            time.sleep(max(0, 0.1 - elapsed))

    except KeyboardInterrupt:
        pass
    finally:
        robot.stop()
        print("Stopped")

In [ ]:
display(sidebyside)
run_line_only(robot)